# BloodBalance — Transfusion Donor Analysis Notebook

**Dataset:** UCI Blood Transfusion Service Center  
**Source:** I-Cheng Yeh (2008), Hsin-Chu City, Taiwan  
**Target window:** March 2007  

> ⚠️ **IMPORTANT DISCLAIMER:**  
> This dataset is **completely separate** from the Indian blood-bank directory.  
> It is historical Taiwanese data. The model below **does not** predict:  
> - Indian blood-bank stock levels  
> - Current donor availability  
> - Real-world blood shortages  
> It is an **educational classification exercise** only.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install plotly pandas scikit-learn --quiet

In [ ]:
# ── Cell 2: Data path setup ───────────────────────────────────────────────────
import os

# Colab: upload or mount Drive
# from google.colab import files; uploaded = files.upload()
# DATA_PATH = 'transfusion_raw.csv'

# Local VS Code
DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'raw', 'transfusion_raw.csv')
print(f'Using: {DATA_PATH}')

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, RocCurveDisplay
)

print('All imports OK')

In [ ]:
# ── Cell 4: Load and rename ───────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]
df.rename(columns={
    'Recency (months)': 'recency_months',
    'Frequency (times)': 'frequency_times',
    'Monetary (c.c. blood)': 'monetary_cc',
    'Time (months)': 'time_months',
    'whether he/she donated blood in March 2007': 'donated',
}, inplace=True)

print(f'Shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print()
print(df.describe())

In [ ]:
# ── Cell 5: Target class distribution ────────────────────────────────────────
print('Target distribution:')
vc = df['donated'].value_counts()
print(vc)
print(f'\nClass imbalance ratio: {vc[0]/vc[1]:.1f}:1 (did not donate : donated)')

fig = go.Figure(go.Bar(
    x=['Did Not Donate (0)', 'Donated (1)'],
    y=[vc[0], vc[1]],
    marker_color=['#E74C3C', '#1ABC9C'],
    text=[vc[0], vc[1]], textposition='outside',
))
fig.update_layout(title='Target Class Distribution (Class Imbalance Present)',
                  yaxis_title='Count', height=380)
fig.show()

In [ ]:
# ── Cell 6: Feature distributions ────────────────────────────────────────────
cols = ['recency_months', 'frequency_times', 'monetary_cc', 'time_months']
labels = ['Recency (months)', 'Frequency (donations)', 'Monetary (cc)', 'Time (months)']
colors = ['#C0392B', '#F39C12', '#1ABC9C', '#3498DB']

fig = make_subplots(rows=2, cols=2, subplot_titles=labels)
for i, (col, color) in enumerate(zip(cols, colors)):
    r, c = divmod(i, 2)
    fig.add_trace(go.Histogram(x=df[col], marker_color=color, opacity=0.8,
                               name=labels[i], showlegend=False), row=r+1, col=c+1)
fig.update_layout(title='RFM Feature Distributions', height=500)
fig.show()

In [ ]:
# ── Cell 7: Correlation ───────────────────────────────────────────────────────
corr = df.corr()
fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r',
                title='Feature Correlation Matrix',
                zmin=-1, zmax=1)
fig.update_layout(height=450)
fig.show()
print('Note: Frequency and Monetary are perfectly correlated (each donation = 250 cc).')

In [ ]:
# ── Cell 8: Scatter — Recency vs Frequency ────────────────────────────────────
df_plot = df.copy()
df_plot['Outcome'] = df_plot['donated'].map({1: 'Donated', 0: 'Did Not Donate'})
fig = px.scatter(df_plot, x='recency_months', y='frequency_times',
                 color='Outcome',
                 color_discrete_map={'Donated': '#1ABC9C', 'Did Not Donate': '#E74C3C'},
                 opacity=0.6,
                 title='Recency vs Frequency by Donation Outcome',
                 labels={'recency_months': 'Recency (months)', 'frequency_times': 'Frequency'})
fig.show()

In [ ]:
# ── Cell 9: Logistic Regression ───────────────────────────────────────────────
feature_cols = ['recency_months', 'frequency_times', 'monetary_cc', 'time_months']
X = df[feature_cols].values
y = df['donated'].values

# Stratified split to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Logistic regression with balanced class weights
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)

print('=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred, target_names=['Did Not Donate', 'Donated']))
print(f'ROC-AUC Score: {roc_auc:.4f}')
print()
print('Why ROC-AUC matters more than accuracy here:')
print('  - Dataset is 76% class 0. A dummy classifier achieves 76% accuracy.')
print('  - ROC-AUC measures discrimination ability regardless of threshold.')
print(f'  - Our ROC-AUC of {roc_auc:.3f} shows meaningful signal above random (0.5).')

In [ ]:
# ── Cell 10: Confusion matrix ─────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig = go.Figure(go.Heatmap(
    z=cm,
    x=['Predicted: No', 'Predicted: Yes'],
    y=['Actual: No', 'Actual: Yes'],
    colorscale='Reds',
    text=cm.astype(str), texttemplate='%{text}',
    showscale=False,
))
fig.update_layout(title='Confusion Matrix', height=350)
fig.show()

In [ ]:
# ── Cell 11: Feature coefficients ────────────────────────────────────────────
coeffs = model.coef_[0]
labels = ['Recency', 'Frequency', 'Monetary', 'Time']
order = np.argsort(coeffs)
colors = ['#1ABC9C' if c >= 0 else '#E74C3C' for c in coeffs[order]]

fig = go.Figure(go.Bar(
    x=coeffs[order], y=[labels[i] for i in order],
    orientation='h', marker_color=colors,
    text=[f'{c:.3f}' for c in coeffs[order]], textposition='outside',
))
fig.update_layout(
    title='Logistic Regression Coefficients (Scaled Features)',
    xaxis_title='Coefficient Value', height=350
)
fig.show()
print()
print('Interpretation (scaled features, balanced weights):')
print('  Negative Recency coeff → more recent donors more likely to donate again.')
print('  Positive Frequency/Monetary → more frequent donors more likely to donate.')
print('  Frequency and Monetary are collinear — interpret jointly, not separately.')

In [ ]:
# ── Cell 12: 5-fold cross-validation ─────────────────────────────────────────
cv_scores = cross_val_score(model, scaler.fit_transform(X), y, cv=5, scoring='roc_auc')
print('5-Fold Cross-Validation ROC-AUC:')
print(f'  Scores : {[round(s, 3) for s in cv_scores]}')
print(f'  Mean   : {cv_scores.mean():.3f}')
print(f'  Std    : {cv_scores.std():.3f}')
print()
print('=== MODEL LIMITATIONS ===')
print('1. Frequency and Monetary are perfectly correlated (250 cc/donation).')
print('2. Dataset from one Taiwanese mobile unit in 2007 — not generalisable.')
print('3. For educational demonstration only. Not for operational use.')